In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("JeremiahZ/roberta-base-sst2")
model = AutoModelForSequenceClassification.from_pretrained("JeremiahZ/roberta-base-sst2")

/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [4]:
#obfuscate model
model = model_obfuscation(model)
print(model)

Obfuscating: roberta.encoder.layer.0.attention.self.query
Obfuscating: roberta.encoder.layer.0.attention.self.key
Obfuscating: roberta.encoder.layer.0.attention.self.value
Obfuscating: roberta.encoder.layer.0.attention.output.dense
Obfuscating: roberta.encoder.layer.0.intermediate.dense
Obfuscating: roberta.encoder.layer.0.output.dense
Obfuscating: roberta.encoder.layer.1.attention.self.query
Obfuscating: roberta.encoder.layer.1.attention.self.key
Obfuscating: roberta.encoder.layer.1.attention.self.value
Obfuscating: roberta.encoder.layer.1.attention.output.dense
Obfuscating: roberta.encoder.layer.1.intermediate.dense
Obfuscating: roberta.encoder.layer.1.output.dense
Obfuscating: roberta.encoder.layer.2.attention.self.query
Obfuscating: roberta.encoder.layer.2.attention.self.key
Obfuscating: roberta.encoder.layer.2.attention.self.value
Obfuscating: roberta.encoder.layer.2.attention.output.dense
Obfuscating: roberta.encoder.layer.2.intermediate.dense
Obfuscating: roberta.encoder.layer.2

In [5]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device='cuda')

In [6]:
# two examples from training set

sentence = "contains no wit , only labored gags"

print(classifier(sentence))

sentence = "are more deeply thought through than in most ` right-thinking ' films"
print(classifier(sentence))

[{'label': 'negative', 'score': 0.9921139478683472}]
[{'label': 'positive', 'score': 0.9955770969390869}]


In [7]:
dataset = load_dataset("glue", "sst2")['validation']
print(dataset)

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 872
})


In [8]:
correct = 0
total = 0

for i in tqdm.tqdm(range(872)):
    result = classifier(dataset[i]['sentence'])
    result = 0 if result[0]['label'] == 'negative' else 1
    if result == dataset[i]['label']:
        correct += 1
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 872/872 [00:08<00:00, 97.16it/s]

correct:816, total:872, accuracy:0.9357798165137615
